# 03 · Mendeley Preprocessing Harmonization

The external evaluation set is reprocessed through the intensity normalization
applied to the training datasets (1-99 percentile stretch to the full 8-bit range),
aligning its photometric characteristics with the training data. Images are
processed in parallel; any unreadable files are skipped and reported, and are
subsequently excluded during packing.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import numpy as np, time, threading
import concurrent.futures as cf
from PIL import Image
from pathlib import Path

PROC_DIR = Path('/content/drive/MyDrive/Master Thesis/mendeley_oa/processed')
pngs = list(PROC_DIR.glob('Mendeley_*.png'))
print(f'{len(pngs):,} Mendeley files')

Mounted at /content/drive
8,260 Mendeley files


## Parallel harmonization

In [2]:
def normalize_uint8(arr):
    arr = arr.astype(np.float32)
    lo, hi = float(np.percentile(arr, 1)), float(np.percentile(arr, 99))
    if hi - lo < 1: lo, hi = float(arr.min()), float(arr.max())
    if hi <= lo: return np.zeros(arr.shape, dtype=np.uint8)
    return np.clip((arr - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)

done=[0]; bad=[]; lock=threading.Lock(); t0=time.time()
def process(p):
    try:
        arr=np.array(Image.open(str(p)).convert('L'))
        Image.fromarray(normalize_uint8(arr)).save(str(p))
        with lock:
            done[0]+=1
            if done[0]%1000==0: print(f'  {done[0]:,}/{len(pngs):,} ({time.time()-t0:.0f}s)',flush=True)
        return True
    except Exception as e:
        with lock: bad.append((Path(p).name, str(e)[:40]))
        return False
with cf.ThreadPoolExecutor(max_workers=16) as ex: list(ex.map(process, pngs))
print(f'Harmonized {done[0]:,}/{len(pngs):,} in {time.time()-t0:.0f}s')
if bad:
    print(f'{len(bad)} unreadable (skipped; dropped at packing):')
    for n,e in bad: print(f'  {n}: {e}')

  1,000/8,260 (277s)
  2,000/8,260 (294s)
  3,000/8,260 (310s)
  4,000/8,260 (327s)
  5,000/8,260 (343s)
  6,000/8,260 (361s)
  7,000/8,260 (379s)
  8,000/8,260 (396s)
Harmonized 8,259/8,260 in 400s
1 unreadable (skipped; dropped at packing):
  Mendeley_val_KL1_9643665R.png: cannot identify image file '/content/dri


## Verification

In [3]:
sample=np.random.RandomState(42).choice(len(pngs), min(100,len(pngs)), replace=False)
means=[]; rng=[]
for i in sample:
    try:
        a=np.array(Image.open(str(pngs[i])).convert('L'),dtype=np.float32)
        means.append(a.mean()); rng.append((a.min(),a.max()))
    except Exception: pass
print(f'Brightness: {np.mean(means):.1f}')
print(f'Range     : [{int(min(r[0] for r in rng))}-{int(max(r[1] for r in rng))}]')

Brightness: 110.8
Range     : [0-255]
